In [1]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta

def run_interactive_fake_door_nice_blue(visitors, conversions, target_cvr):
    # 1. Bayesian Update (The Math)
    prior_alpha = 1
    prior_beta = 1
    
    post_alpha = prior_alpha + conversions
    post_beta = prior_beta + (visitors - conversions)
    
    # 2. Calculate Statistics
    mean_rate = post_alpha / (post_alpha + post_beta)
    
    # Calculate 95% Credible Interval
    ci_lower = beta.ppf(0.025, post_alpha, post_beta) 
    ci_upper = beta.ppf(0.975, post_alpha, post_beta) 
    
    # 3. Generate Data for Plotting
    x_max = max(mean_rate * 3, target_cvr * 2, ci_upper * 1.5)
    x = np.linspace(0, x_max, 2000)
    y = beta.pdf(x, post_alpha, post_beta)
    
    # 4. Create Interactive Plotly Graph
    fig = go.Figure()

    # Define Theme Colors
    primary_blue = '#1f77b4' # Standard light blue
    fill_blue = 'rgba(31, 119, 180, 0.3)' # Semi-transparent blue

    # A. The Main Curve
    fig.add_trace(go.Scatter(
        x=x, y=y,
        mode='lines',
        name='Probability Curve',
        line=dict(color=primary_blue, width=3)
    ))

    # B. The Highlighted 95% Confidence Area
    mask = (x >= ci_lower) & (x <= ci_upper)
    fig.add_trace(go.Scatter(
        x=x[mask], y=y[mask],
        mode='lines',
        fill='tozeroy', 
        name='95% Confidence Range',
        line=dict(width=0),
        fillcolor=fill_blue
    ))

    # C. Vertical Lines & Labels
    # Target Line (Red)
    fig.add_vline(x=target_cvr, line_width=3, line_dash="solid", line_color="#d62728", 
                  annotation_text="Target Viability", annotation_position="top left", annotation_font=dict(color="#d62728"))

    # Lower Bound Line
    fig.add_vline(x=ci_lower, line_width=2, line_dash="dash", line_color=primary_blue,
                  annotation_text=f"Lower: {ci_lower:.2%}", annotation_position="bottom right", annotation_font=dict(color=primary_blue))

    # Upper Bound Line
    fig.add_vline(x=ci_upper, line_width=2, line_dash="dash", line_color=primary_blue,
                  annotation_text=f"Upper: {ci_upper:.2%}", annotation_position="bottom left", annotation_font=dict(color=primary_blue))

    # Mean Line
    fig.add_vline(x=mean_rate, line_width=2, line_dash="dot", line_color="#005b96")

    # 5. Layout Formatting
    fig.update_layout(
        title=dict(
            text=f"<b>Bayesian Fake Door Test</b><br>N={visitors}, Conv={conversions} | Mean Est: {mean_rate:.2%}",
            font=dict(size=20)
        ),
        xaxis_title="Conversion Rate",
        yaxis_title="Likelihood (Probability Density)",
        xaxis=dict(tickformat=".1%", showgrid=True, gridcolor='#eee'),
        yaxis=dict(showgrid=True, gridcolor='#eee'),
        template="plotly_white",
        showlegend=True,
        legend=dict(y=0.99, x=0.99, bgcolor="rgba(255,255,255,0.8)"),
        font=dict(family="Arial, sans-serif"),
        plot_bgcolor='rgba(0,0,0,0)'
    )
    
    # Add text label
    fig.add_annotation(
        x=mean_rate, y=max(y)/2.5,
        text="95% Confidence<br>Interval",
        showarrow=False,
        font=dict(color=primary_blue, size=12, weight='bold')
    )

    fig.show()

# ==========================================
# ENTER YOUR DATA HERE & RUN
# ==========================================
run_interactive_fake_door_nice_blue(visitors=50, conversions=3, target_cvr=0.01)